#About

This notebook contains code for evaluating the gemma3-4b model fine-tuned for text summarization.

Evaluation is done on the texts that are longer than 16k characters. The shorter texts are used in training.

Articles are summarized using hierarchical summarization.

Rouge, BERTscore, chrF++ metrics are reported.



#Constants

In [ ]:
DATA_PATH = "drive/MyDrive/summaries-data/LITEKO_SUMMARIES"
MODEL_NAME = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit"

MODEL_PATH = "drive/MyDrive/fine-tuned-summaries/gemma3-4b"
IS_FINETUNED = True

GENERATED_SUMMARIES_PATH = "drive/MyDrive/fine-tuned-summaries/generated_summaries/gemma3-4b-finetuned(2)"
TOKEN_COUNT_PER_CHUNK = 3072 * 2
MAX_NEW_TOKENS = 3072
MIN_NEW_TOKENS = 64

ARTICLE_LENGTH_MIN_CUTOFF = 16000

EVAL_SAVE_PATH = "drive/MyDrive/fine-tuned-summaries/generated_summaries/EVALS.csv"

#Setup

In [ ]:
%autosave 60

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
hf_token = "<hf_token>"

from huggingface_hub import login
login(token=hf_token)

In [ ]:
import wandb

In [ ]:
wandb.login(key="<wandb_key>")

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
!pip install rouge-score
!pip install bert-score
!pip install sacrebleu
!python -m spacy download lt_core_news_lg

In [ ]:
SYSTEM_PROMPT = """You are a lawyer.
You will be given a text and you will summarize it.
The summary MUST be in lithuanian.
The summary MUST be a continuous text without any formatting.
The summary MIGHT have multiple paragraphs."""

In [ ]:
from unsloth import FastLanguageModel

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
import pandas as pd
from tqdm.auto import tqdm
from peft import LoraConfig, get_peft_model
import spacy
from datasets import Dataset, load_from_disk, concatenate_datasets
import numpy as np
from rouge_score import rouge_scorer
from unsloth.chat_templates import get_chat_template
from bert_score import BERTScorer

import time
from datetime import datetime
import os

import sacrebleu

In [ ]:
nlp = spacy.load("lt_core_news_lg")

def split_into_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]

#Load data

In [ ]:
dataset = load_from_disk(DATA_PATH)
dataset

In [ ]:
test_dataset = dataset.filter(lambda row: len(row['article']) > ARTICLE_LENGTH_MIN_CUTOFF)
test_dataset

In [ ]:
def get_dataset_summary(dataset):
  temp_dataset = dataset.map(
      lambda example: {
          "article_length": len(example["article"]),
          "summary_length": len(example["summary"])
      }
  )

  article_lengths = temp_dataset["article_length"]
  summary_lengths = temp_dataset["summary_length"]

  percentiles = [0, 25, 50, 75, 90, 100]

  article_percentiles = np.percentile(article_lengths, percentiles)
  summary_percentiles = np.percentile(summary_lengths, percentiles)

  for p, a, s in zip(percentiles, article_percentiles, summary_percentiles):
      print(f"{p}th percentile - article: {a:.1f}, summary: {s:.1f}")

In [ ]:
get_dataset_summary(test_dataset)

#Load model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 10000,
    dtype = torch.bfloat16,
    load_in_4bit = True,
)

In [ ]:
FastLanguageModel.for_inference(model)

In [ ]:
print(next(model.parameters()).dtype)
print(model.device)

In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

#Generation of hierarchical summaries

In [ ]:
def build_prompt(text):
  messages = [
      {"role": "system", "content": SYSTEM_PROMPT},
      {"role": "user", "content": text}
  ]

  prompt = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
  ).removeprefix("<bos>")

  return prompt

In [ ]:
def extract_summary(generated):
  return generated.split("model")[1].strip("\n")

In [ ]:
def generate_batch(tokenized, model):
  response = model.generate(
      **tokenized,
      max_new_tokens=MAX_NEW_TOKENS,
      min_new_tokens=MIN_NEW_TOKENS,
      temperature = 1.0, top_p = 0.95, top_k = 64,
      use_cache=True,
      )

  generated_texts = tokenizer.batch_decode(response, skip_special_tokens=True)

  summaries = [extract_summary(generated_text) for generated_text in generated_texts]

  return summaries

In [ ]:
def generate_summaries_for_chunks(texts, model):

  prompts = [build_prompt(text) for text in texts]

  tokenized = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")

  print(f"Tokenized shape {tokenized['input_ids'].shape}")
  print("generating...")

  generated_batch = generate_batch(tokenized, model)

  return generated_batch

In [ ]:
def chunk_text(text, max_tokens):
  sentences = split_into_sentences(text)

  chunks = []
  current_chunk = []
  current_length = 0

  for sentence in sentences:
      tokenized_len = len(tokenizer.tokenizer.encode(sentence, add_special_tokens=False))
      if current_length + tokenized_len > max_tokens:
          if current_chunk:
              chunks.append(" ".join(current_chunk))
          current_chunk = [sentence]
          current_length = tokenized_len
      else:
          current_chunk.append(sentence)
          current_length += tokenized_len

  # Append last chunk
  if current_chunk:
      chunks.append(" ".join(current_chunk))

  return chunks

In [ ]:
BATCH_SIZE = 2

def generate_summary(text, model, max_tokens):

    chunks = chunk_text(text, max_tokens)
    generated_texts = []

    for batch_idx in range(0, len(chunks), BATCH_SIZE):
      batch = chunks[batch_idx : batch_idx + BATCH_SIZE]
      print(f"Batch #{batch_idx // BATCH_SIZE}, batch size = {len(batch)}")
      summaries = generate_summaries_for_chunks(batch, model)
      generated_texts.extend(summaries)

    return " ".join(generated_texts)

In [ ]:
test_dataset

In [ ]:
start_time = time.time()

generated_summaries = []

for idx, article in tqdm(list(enumerate(test_dataset["article"]))):
    print(f"article #{idx}. length {len(article)}")
    summary = generate_summary(article, model, TOKEN_COUNT_PER_CHUNK)
    generated_summaries.append(summary)
    print()

end_time = time.time()
duration_seconds = end_time - start_time

In [ ]:
test_dataset = test_dataset.add_column("generated_summary", generated_summaries)
test_dataset

In [ ]:
test_dataset.save_to_disk(GENERATED_SUMMARIES_PATH)

#Eval metrics

In [ ]:
eval_dataset = test_dataset
eval_dataset

Rouge evals

In [ ]:
rouge_types = ['rouge1', 'rouge2', 'rougeL']
scorer = rouge_scorer.RougeScorer(rouge_types, use_stemmer=False)

In [ ]:
def evaluate_rouge(generated_summary, ground_summary):
    score = scorer.score(ground_summary, generated_summary)

    results = {}
    for s in rouge_types:
        results[f"{s}_precision"] = round(score[s].precision, 4)
        results[f"{s}_recall"] = round(score[s].recall, 4)
        results[f"{s}_f1"] = round(score[s].fmeasure, 4)

    return results

In [ ]:
def add_rouge_evals(example):
    scores = evaluate_rouge(example["generated_summary"], example["summary"])
    return scores

In [ ]:
eval_dataset = eval_dataset.map(add_rouge_evals)
eval_dataset

In [ ]:
def get_rouge_eval(rouge_metric, dataset):
  rouge_eval = {}
  metric_column = dataset[rouge_metric]

  rouge_eval[f"{rouge_metric}_mean"] = round(np.mean(metric_column), 4)
  # rouge_eval[f"{rouge_metric}_std"] = round(np.std(metric_column), 4)
  # rouge_eval[f"{rouge_metric}_median"] = round(np.median(metric_column), 4)
  # rouge_eval[f"{rouge_metric}_min"] = round(np.min(metric_column), 4)
  # rouge_eval[f"{rouge_metric}_max"] = round(np.max(metric_column), 4)

  return rouge_eval

def get_rouge_evals(dataset):
  rouge_evals = {}
  for r in rouge_types:

    precision_metric = f"{r}_precision"
    precision_eval = get_rouge_eval(precision_metric, dataset)

    recall_metric = f"{r}_recall"
    recall_eval = get_rouge_eval(recall_metric, dataset)

    f1_metric = f"{r}_f1"
    f1_eval = get_rouge_eval(f1_metric, dataset)

    rouge_evals.update(f1_eval | precision_eval | recall_eval)

  return rouge_evals

In [ ]:
rouge_evals = get_rouge_evals(eval_dataset)
rouge_evals

chrF++ evals

In [ ]:
chrf = sacrebleu.CHRF(word_order=2)

In [ ]:
def evaluate_chrf(generated_summary, ground_summary):
    score = chrf.corpus_score([generated_summary], [[ground_summary]])
    return round(score.score, 4)

In [ ]:
def add_chrf_evals(example):
    chrf = evaluate_chrf(example["generated_summary"], example["summary"])
    return {"chrF++": chrf}

In [ ]:
eval_dataset = eval_dataset.map(add_chrf_evals)
eval_dataset

In [ ]:
def get_chrf_standard(dataset):
  ground_summaries = dataset["summary"]
  ground_summaries_nested = [[summary] for summary in ground_summaries]
  chrf_standard = chrf.corpus_score(dataset["generated_summary"], ground_summaries_nested)
  return chrf_standard.score

In [ ]:
chrf_evals = {
    "chrF++ (corpus score - standard)": round(get_chrf_standard(eval_dataset), 4),
    "chrF++ (macro average)": round(np.mean(eval_dataset["chrF++"]), 4),
}

chrf_evals

Bert score evals

In [ ]:
bert_scorer = BERTScorer(model_type="Justelioo/LT-BERT-SCORE", num_layers=22)

In [ ]:
def add_bert_score(example):
  P, R, F1 = bert_scorer.score([example["generated_summary"]], [example["summary"]], verbose=False)
  return {
      "bert_score_f1": F1.item(),
      "bert_score_P": P.item(),
      "bert_score_R": R.item(),
    }

In [ ]:
eval_dataset = eval_dataset.map(add_bert_score, batched=False)
eval_dataset

In [ ]:
bert_score_evals = {
    "bert_score_f1_mean": round(np.mean(eval_dataset["bert_score_f1"]), 4),
    "bert_score_precision_mean": round(np.mean(eval_dataset["bert_score_P"]), 4),
    "bert_score_recall_mean": round(np.mean(eval_dataset["bert_score_R"]), 4),
}

bert_score_evals

Length data

In [ ]:
def add_length_data(example):
    return {
        "article_length": len(example["article"]),
        "summary_length": len(example["summary"]),
        "generated_summary_length": len(example["generated_summary"]),
    }

In [ ]:
eval_dataset = eval_dataset.map(add_length_data)
eval_dataset

In [ ]:
lengths_data = {
    "article_length_mean": np.mean(eval_dataset["article_length"]),
    "summary_length_mean": np.mean(eval_dataset["summary_length"]),
    "generated_summary_length_mean": np.mean(eval_dataset["generated_summary_length"]),
    "summary_length_mean_%_of_article_length_mean": round(np.mean(eval_dataset["summary_length"]) / np.mean(eval_dataset["article_length"]), 4),
    "generated_summary_length_mean_%_of_article_length_mean": round(np.mean(eval_dataset["generated_summary_length"]) / np.mean(eval_dataset["article_length"]), 4),
}

lengths_data

Meta data

In [ ]:
meta_data = {
    "model_name": MODEL_NAME,
    "finetuned": IS_FINETUNED,
    "log_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "eval_sample_size": len(eval_dataset),
    "eval_duration_seconds": round(duration_seconds, 4),
    "eval_duration_minutes": round(duration_seconds / 60, 4),
    "eval_duration_hours": round(duration_seconds / 3600, 4),
    "token_count_per_chunk": TOKEN_COUNT_PER_CHUNK,
    "article_length_min_cutoff": ARTICLE_LENGTH_MIN_CUTOFF,
    "max_new_tokens": MAX_NEW_TOKENS,
    "min_new_tokens": MIN_NEW_TOKENS,
    "chunks_per_batch": BATCH_SIZE,
    "model_path": MODEL_PATH,
    "generated_summaries_path": GENERATED_SUMMARIES_PATH,
}

meta_data

In [ ]:
all_evals = meta_data | lengths_data | rouge_evals | chrf_evals | bert_score_evals
all_evals

In [ ]:
def log_evals(file_name, results_dict):
    # 1. Convert the new result into a single-row DataFrame
    new_data = pd.DataFrame([results_dict])

    if os.path.exists(file_name):
        # 2. Read existing data
        existing_df = pd.read_csv(file_name)

        # 3. Combine them - Pandas automatically adds 'NaN' for missing values
        combined_df = pd.concat([existing_df, new_data], ignore_index=True, sort=False)
    else:
        combined_df = new_data

    # 4. Save back to CSV
    combined_df.to_csv(file_name, index=False)
    print(f"Logged to {file_name}. Total rows: {len(combined_df)}")

In [ ]:
log_evals(EVAL_SAVE_PATH, all_evals)

In [ ]:
eval_dataset['article'][0]

In [ ]:
eval_dataset['summary'][0]

In [ ]:
eval_dataset['generated_summary'][0]